In [ ]:
import cv2
import mediapipe as mp
from twilio.rest import Client

print("All libraries are working ")

TWILIO_SID = 'YOUR_TWILIO_SID'
TWILIO_AUTH = 'YOUR_TWILIO_AUTH_TOKEN'
FROM_PHONE = 'YOUR_TWILIO_PHONE'
TO_PHONE = 'DIRECTOR_PHONE_NUMBER'

client = Client(TWILIO_SID, TWILIO_AUTH)

def send_sms_alert():
    message = client.messages.create(
        body="ALERT : An employee appears to be sleeping during work hours.",
        from_=FROM_PHONE,
        to=TO_PHONE
    )
    print("SMS sent:", message.sid)

def get_EAR(landmarks, eye_indices):
    p1 = landmarks[eye_indices[1]]
    p2 = landmarks[eye_indices[5]]
    p3 = landmarks[eye_indices[2]]
    p4 = landmarks[eye_indices[4]]
    p5 = landmarks[eye_indices[0]]
    p6 = landmarks[eye_indices[3]]

    vertical1 = ((p2.x - p4.x)**2 + (p2.y - p4.y)**2)**0.5
    vertical2 = ((p3.x - p5.x)**2 + (p3.y - p5.y)**2)**0.5
    horizontal = ((p1.x - p6.x)**2 + (p1.y - p6.y)**2)**0.5

    ear = (vertical1 + vertical2) / (2.0 * horizontal)
    return ear

EYE_AR_THRESHOLD = 0.22
EYE_AR_CONSEC_FRAMES = 50  

COUNTER = 0
ALERT_SENT = False

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)
LEFT_EYE = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]

cap = cv2.VideoCapture(0)

print("[INFO] Starting camera...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)

    if results.multi_face_landmarks:
        mesh_points = results.multi_face_landmarks[0].landmark
        left_ear = get_EAR(mesh_points, LEFT_EYE)
        right_ear = get_EAR(mesh_points, RIGHT_EYE)
        ear = (left_ear + right_ear) / 2.0

        if ear < EYE_AR_THRESHOLD:
            COUNTER += 1
            if COUNTER >= EYE_AR_CONSEC_FRAMES and not ALERT_SENT:
                print("[ALERT] Sleep detected. Sending SMS...")
                send_sms_alert()
                ALERT_SENT = True
        else:
            COUNTER = 0
            ALERT_SENT = False

    cv2.imshow('Monitoring - Press Q to Quit', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


ModuleNotFoundError: No module named 'cv2'